# DSA 4020A — NLP Term Project
# Week 1: Data Collection & Curation — Security & Safety Domain (Web Scraping)

**Course:** DSA 4020A Natural Language Processing (Dr. Edward Ombui)
**Project:** Machine Translation of Public Service Announcements (PSAs) in Kenya
**This notebook covers:** Sub-Objective 1 — Web scraping pipeline for the **Security & Safety**
domain only. The **manual curation** portion (native-speaker sourced / hand-collected PSAs,
and the Kiswahili/Ekegusii/Dholuo/Somali translations) is intentionally left as a **placeholder**
here — those columns are created empty and are meant to be filled in a later, separate step
(either manually or with an MT model in Week 3).

**Target:** ~2,000 rows for the Security & Safety domain (the remaining domains — Health,
Agriculture, Education, Governance — are handled in separate notebooks/runs so the whole
project reaches the >=5,000-sentence requirement).

---

### Important note on running this notebook

This notebook was authored in a sandboxed environment whose outbound network access is
restricted to package registries (PyPI, npm, GitHub) — it **cannot reach `ntsa.go.ke`,
`nationalpolice.go.ke`, `redcross.or.ke`, etc. from here**, so the scraping cells below have
**not been executed against the live sites**. The code is written to be run by you in a normal
internet-connected environment (Google Colab, your laptop, or a USIU lab machine).

Because I couldn't render the live HTML of every source, I built the scrapers around a
**generic, configurable extraction pattern** (`SOURCES` list below) rather than brittle
site-specific CSS selectors guessed from memory. **The first time you run this, keep
`DEBUG = True`** at the top of the pipeline — it will print the raw HTML structure of each
listing page so you can quickly see which CSS selector actually holds the headline/link on
each site, and adjust it. This is normal for real-world scraping projects and is exactly the
kind of "challenges faced" detail your Week 1 report is supposed to document.

### Real sources this notebook targets for Security & Safety
| # | Source | Type | URL |
|---|--------|------|-----|
| 1 | National Transport & Safety Authority (NTSA) | Government agency | https://www.ntsa.go.ke/news/1 |
| 2 | National Police Service (NPS) Kenya | Government agency | https://www.nationalpolice.go.ke/allnews |
| 3 | National Police Service Commission | Government agency | https://www.npsc.go.ke/press-releases/ |
| 4 | Kenya Red Cross Society | NGO / humanitarian | https://redcross.or.ke/ |
| 5 | Ministry of Roads and Transport | Government ministry | https://www.transport.go.ke |
| 6 | Kenya News Agency (KNA) | State media wire | https://www.kenyanews.go.ke |
| 7 | Directorate of Criminal Investigations (DCI) | Government agency | https://www.dci.go.ke |
| 8 | ReliefWeb (Kenya disaster updates) | NGO/UN aggregator | https://reliefweb.int/country/ken |
| 9 | e-Citizen Advisories | Government | https://www.ecitizen.go.ke |
| 10 | Communications Authority of Kenya / KE-CIRT (cybersecurity alerts) | Government agency | https://www.ke-cirt.go.ke |

That's 10 sources, meeting the >=10-source milestone requirement.


In [ ]:
# 1. Setup & installs
# Run this cell first (in Colab: !pip install ..., locally: use your venv / requirements.txt)

import sys
!{sys.executable} -m pip install -q requests beautifulsoup4 lxml langdetect pandas tqdm python-dateutil
print("Packages ready.")


Packages ready.


In [ ]:
# 2. Imports
import re
import time
import json
import random
import urllib.robotparser as robotparser
from datetime import datetime
from urllib.parse import urljoin, urlparse

import requests
from bs4 import BeautifulSoup
import pandas as pd
from dateutil import parser as dateparser
from langdetect import detect, DetectorFactory
from tqdm.auto import tqdm

DetectorFactory.seed = 42  # deterministic langdetect

pd.set_option("display.max_colwidth", 120)


In [ ]:
# 3. Global config
DEBUG = True                      # print HTML structure while inspecting/tuning selectors
REQUEST_TIMEOUT = 15              # seconds
MIN_DELAY, MAX_DELAY = 1.5, 3.5   # polite rate-limiting: random delay between requests (seconds)
MAX_PAGES_PER_SOURCE = 15         # pagination depth cap per source (raise once selectors are confirmed)
TARGET_ROWS = 2000                # Security & Safety target for this notebook
DOMAIN = "Security & Safety"

HEADERS = {
    "User-Agent": (
        "USIU-Africa-DSA4020A-ResearchBot/1.0 "
        "(Academic NLP project - Kenyan PSA dataset collection; "
        "contact: student research project, non-commercial; "
        "respects robots.txt and rate limits)"
    )
}

SECURITY_SAFETY_SUBCATEGORIES = {
    "Public Safety Awareness": [
        "road safety", "fire", "flood", "disaster", "evacuat", "drought", "landslide",
        "accident", "crash", "safety advisory", "warn", "advisory", "alert", "rescue",
    ],
    "Crime Prevention": [
        "crime", "corruption", "community policing", "report a crime", "hotline",
        "arrest", "wanted", "burglary", "theft", "robbery", "gang",
    ],
    "National Security": [
        "terror", "border security", "national security", "peace-building", "peacebuilding",
        "counter-terrorism", "counterterrorism", "extremis",
    ],
    "Gender-Based Violence": [
        "gender-based violence", "gbv", "domestic violence", "sexual assault", "survivor",
        "femicide", "child protection",
    ],
    "Cybersecurity": [
        "cyber", "online scam", "data privacy", "phishing", "fraud alert", "mpesa fraud",
        "sim swap", "data protection", "online safety",
    ],
}

PSA_ACTION_HINTS = [
    "urge", "warn", "advis", "remind", "must", "should", "please", "avoid", "ensure",
    "report to", "call ", "dial ", "register", "apply before", "deadline", "comply",
    "hotline", "alert", "notice", "public is informed", "members of the public",
]


In [ ]:
# 4. Source configuration
# NOTE: `list_selector` / `link_selector` / `title_selector` / `date_selector` /
# `body_selector` are best-effort guesses based on typical WordPress/Gov-CMS structures.
# RUN WITH DEBUG=True FIRST and adjust these before a full scrape -- see `inspect_source()`.

SOURCES = [
    {
        "name": "NTSA",
        "base_url": "https://www.ntsa.go.ke",
        "listing_url_template": "https://www.ntsa.go.ke/news/{page}",
        "start_page": 1,
        "list_selector": "article, .post, .news-item, li.news",
        "link_selector": "a",
        "title_selector": "h1, h2, h3, .entry-title",
        "date_selector": "time, .entry-date, .post-date",
        "body_selector": "div.entry-content, div.post-content, article",
    },
    {
        "name": "National Police Service (NPS)",
        "base_url": "https://www.nationalpolice.go.ke",
        "listing_url_template": "https://www.nationalpolice.go.ke/allnews?page={page}",
        "start_page": 0,
        "list_selector": "article, .views-row, .node--type-article",
        "link_selector": "a",
        "title_selector": "h1, h2, h3, .field--name-title",
        "date_selector": "time, .field--name-created",
        "body_selector": "div.field--name-body, article",
    },
    {
        "name": "National Police Service Commission (NPSC)",
        "base_url": "https://www.npsc.go.ke",
        "listing_url_template": "https://www.npsc.go.ke/press-releases/page/{page}/",
        "start_page": 1,
        "list_selector": "article, .post",
        "link_selector": "a",
        "title_selector": "h1, h2, .entry-title",
        "date_selector": "time, .entry-date",
        "body_selector": "div.entry-content",
    },
    {
        "name": "Kenya Red Cross Society",
        "base_url": "https://redcross.or.ke",
        "listing_url_template": "https://redcross.or.ke/news/page/{page}/",
        "start_page": 1,
        "list_selector": "article, .post, .news-card",
        "link_selector": "a",
        "title_selector": "h1, h2, .entry-title",
        "date_selector": "time, .entry-date",
        "body_selector": "div.entry-content",
    },
    {
        "name": "Ministry of Roads and Transport",
        "base_url": "https://www.transport.go.ke",
        "listing_url_template": "https://www.transport.go.ke/press-statements/page/{page}/",
        "start_page": 1,
        "list_selector": "article, .post",
        "link_selector": "a",
        "title_selector": "h1, h2, .entry-title",
        "date_selector": "time, .entry-date",
        "body_selector": "div.entry-content",
    },
    {
        "name": "Kenya News Agency (KNA)",
        "base_url": "https://www.kenyanews.go.ke",
        "listing_url_template": "https://www.kenyanews.go.ke/category/security/page/{page}/",
        "start_page": 1,
        "list_selector": "article, .post",
        "link_selector": "a",
        "title_selector": "h1, h2, .entry-title",
        "date_selector": "time, .entry-date",
        "body_selector": "div.entry-content",
    },
    {
        "name": "Directorate of Criminal Investigations (DCI)",
        "base_url": "https://www.dci.go.ke",
        "listing_url_template": "https://www.dci.go.ke/news/page/{page}/",
        "start_page": 1,
        "list_selector": "article, .post",
        "link_selector": "a",
        "title_selector": "h1, h2, .entry-title",
        "date_selector": "time, .entry-date",
        "body_selector": "div.entry-content",
    },
    {
        "name": "ReliefWeb Kenya",
        "base_url": "https://reliefweb.int",
        "listing_url_template": "https://reliefweb.int/updates?advanced-search=%28C107%29&page={page}",
        "start_page": 0,
        "list_selector": "article, .rw-river-article",
        "link_selector": "a",
        "title_selector": "h1, h2, .rw-river-article__title",
        "date_selector": "time",
        "body_selector": "div.rw-article-body, article",
    },
    {
        "name": "e-Citizen Advisories",
        "base_url": "https://www.ecitizen.go.ke",
        "listing_url_template": "https://www.ecitizen.go.ke/news?page={page}",
        "start_page": 1,
        "list_selector": "article, .news-item",
        "link_selector": "a",
        "title_selector": "h1, h2",
        "date_selector": "time",
        "body_selector": "article",
    },
    {
        "name": "KE-CIRT Cybersecurity Alerts",
        "base_url": "https://www.ke-cirt.go.ke",
        "listing_url_template": "https://www.ke-cirt.go.ke/index.php/alerts-and-advisories/page/{page}/",
        "start_page": 1,
        "list_selector": "article, .post",
        "link_selector": "a",
        "title_selector": "h1, h2, .entry-title",
        "date_selector": "time, .entry-date",
        "body_selector": "div.entry-content",
    },
]

print(f"{len(SOURCES)} sources configured.")


10 sources configured.


In [ ]:
import re
import time
import json
import random
import urllib.robotparser as robotparser
from datetime import datetime
from urllib.parse import urljoin, urlparse

import requests
from bs4 import BeautifulSoup
import pandas as pd
from dateutil import parser as dateparser
from langdetect import detect, DetectorFactory
from tqdm.auto import tqdm

# 5. Politeness helpers: robots.txt check + rate-limited GET

_robots_cache = {}

def allowed_by_robots(url, user_agent="*"):
    """Check robots.txt before hitting a URL. On error, proceed cautiously (logged) rather
    than silently skipping the whole source -- most gov sites simply don't set robots.txt."""
    parsed = urlparse(url)
    root = f"{parsed.scheme}://{parsed.netloc}"
    if root not in _robots_cache:
        rp = robotparser.RobotFileParser()
        rp.set_url(urljoin(root, "/robots.txt"))
        try:
            rp.read()
            _robots_cache[root] = rp
        except Exception:
            _robots_cache[root] = None
    rp = _robots_cache[root]
    if rp is None:
        return True
    try:
        return rp.can_fetch(user_agent, url)
    except Exception:
        return True

def polite_get(url, session):
    if not allowed_by_robots(url):
        print(f"  [BLOCKED by robots.txt] {url}")
        return None
    try:
        # Added verify=False to bypass SSL certificate verification for debugging.
        # WARNING: Disabling SSL verification can pose security risks.
        resp = session.get(url, headers=HEADERS, timeout=REQUEST_TIMEOUT, verify=False)
        resp.raise_for_status()
        time.sleep(random.uniform(MIN_DELAY, MAX_DELAY))
        return resp
    except requests.RequestException as e:
        print(f"  [request failed] {url} -> {e}")
        return None


In [ ]:
# 6. Debug helper -- inspect a source's raw HTML structure before trusting the selectors
def inspect_source(source, page=None, session=None):
    session = session or requests.Session()
    page = source["start_page"] if page is None else page
    url = source["listing_url_template"].format(page=page)
    print(f"Fetching: {url}")
    resp = polite_get(url, session)
    if resp is None:
        print("  -> could not fetch. Check the URL manually in a browser.")
        return None
    soup = BeautifulSoup(resp.text, "lxml")
    items = soup.select(source["list_selector"])
    print(f"  -> {len(items)} elements matched list_selector='{source['list_selector']}'")
    if items:
        print("  -> sample element (first match), truncated:\n")
        print(str(items[0])[:800])
    else:
        print("  -> NO MATCHES. Open the URL in a browser, right-click a headline, "
              "'Inspect', and update list_selector/link_selector/title_selector for this source.")
    return soup

# Example (run manually, one source at a time, before the full crawl):
# _ = inspect_source(SOURCES[0])


In [ ]:
# 7. Extraction logic: listing page -> article links, article page -> clean text
def extract_links_from_listing(soup, source):
    links = set()
    containers = soup.select(source["list_selector"]) or [soup]
    for container in containers:
        for a in container.select(source["link_selector"]):
            href = a.get("href")
            if not href:
                continue
            full = urljoin(source["base_url"], href)
            if urlparse(full).netloc == urlparse(source["base_url"]).netloc:
                links.add(full.split("#")[0])
    return links


def clean_text(text):
    return re.sub(r"\s+", " ", text or "").strip()


def guess_date(soup, source):
    el = soup.select_one(source["date_selector"])
    if not el:
        return None
    raw = el.get("datetime") or el.get_text()
    try:
        return dateparser.parse(raw, fuzzy=True).date().isoformat()
    except Exception:
        return None


def extract_article(url, source, session):
    resp = polite_get(url, session)
    if resp is None:
        return None
    soup = BeautifulSoup(resp.text, "lxml")

    title_el = soup.select_one(source["title_selector"])
    title = clean_text(title_el.get_text()) if title_el else None

    body_el = soup.select_one(source["body_selector"])
    if body_el:
        paragraphs = [clean_text(p.get_text()) for p in body_el.find_all(["p", "li"])]
        paragraphs = [p for p in paragraphs if len(p) > 20]
        body = " ".join(paragraphs)
    else:
        body = None

    date = guess_date(soup, source)

    if not title and not body:
        return None

    return {"title": title, "body": body, "date": date, "url": url}


In [ ]:
# 8. PSA-style sentence extraction
# A full news article is NOT a PSA. A PSA is a short, directive, action-oriented statement.
# We split each scraped article into candidate sentences and keep the ones that read like
# genuine PSAs (contain an action/advisory cue and are within a plausible PSA length),
# instead of dumping whole articles into the dataset.

SENTENCE_SPLIT_RE = re.compile(r"(?<=[.!?])\s+")

def looks_like_psa(sentence):
    s = sentence.lower()
    if not (25 <= len(sentence) <= 320):
        return False
    return any(hint in s for hint in PSA_ACTION_HINTS)


def classify_subcategory(text):
    t = text.lower()
    best, best_hits = "Public Safety Awareness", 0
    for subcat, keywords in SECURITY_SAFETY_SUBCATEGORIES.items():
        hits = sum(1 for kw in keywords if kw in t)
        if hits > best_hits:
            best, best_hits = subcat, hits
    return best


def is_english(text):
    try:
        return detect(text) == "en"
    except Exception:
        return False


def candidate_psas_from_article(article, source_name):
    if not article or not article.get("title"):
        return []
    text_blob = " ".join(filter(None, [article["title"], article.get("body") or ""]))
    sentences = SENTENCE_SPLIT_RE.split(text_blob)
    # Always include the (cleaned) headline itself as a PSA-style candidate too,
    # since Kenyan gov headlines are often already PSA-shaped ("NTSA warns motorists...").
    candidates = [article["title"]] + [s for s in sentences if looks_like_psa(s)]

    rows, seen = [], set()
    for c in candidates:
        c = clean_text(c)
        key = c.lower()
        if len(c) < 25 or key in seen:
            continue
        if not is_english(c):
            continue
        seen.add(key)
        rows.append({
            "English": c,
            "Sub_Category": classify_subcategory(c),
            "Source": source_name,
            "Date": article.get("date"),
            "URL": article["url"],
        })
    return rows


In [ ]:
# 9. Main crawl loop
def scrape_source(source, target_rows_remaining, session):
    collected = []
    page = source["start_page"]
    pages_tried = 0
    seen_articles = set()

    print(f"\n=== Scraping: {source['name']} ===")
    while pages_tried < MAX_PAGES_PER_SOURCE and len(collected) < target_rows_remaining:
        url = source["listing_url_template"].format(page=page)
        resp = polite_get(url, session)
        pages_tried += 1
        page += 1
        if resp is None:
            break

        soup = BeautifulSoup(resp.text, "lxml")
        links = extract_links_from_listing(soup, source)
        if DEBUG:
            print(f"  page {page-1}: {len(links)} candidate links")
        if not links:
            break  # likely reached the end, or selector needs fixing (see inspect_source)

        for link in tqdm(links, leave=False, desc=f"{source['name']} p{page-1}"):
            if link in seen_articles:
                continue
            seen_articles.add(link)
            article = extract_article(link, source, session)
            rows = candidate_psas_from_article(article, source["name"])
            collected.extend(rows)
            if len(collected) >= target_rows_remaining:
                break

    print(f"  -> {len(collected)} PSA-style rows collected from {source['name']}")
    return collected


def run_full_crawl():
    session = requests.Session()
    all_rows = []
    for source in SOURCES:
        remaining = TARGET_ROWS - len(all_rows)
        if remaining <= 0:
            break
        try:
            rows = scrape_source(source, remaining, session)
            all_rows.extend(rows)
        except Exception as e:
            print(f"  [source failed: {source['name']}] {e}")
            continue
    return all_rows

# Uncomment to run the full crawl (do this in an internet-connected environment):
raw_rows = run_full_crawl()
print(f"\nTotal raw rows collected: {len(raw_rows)}")



=== Scraping: NTSA ===
  [request failed] https://www.ntsa.go.ke/news/1 -> HTTPSConnectionPool(host='www.ntsa.go.ke', port=443): Max retries exceeded with url: /news/1 (Caused by SSLError(SSLCertVerificationError(1, '[SSL: CERTIFICATE_VERIFY_FAILED] certificate verify failed: unable to get local issuer certificate (_ssl.c:1010)')))
  -> 0 PSA-style rows collected from NTSA

=== Scraping: National Police Service (NPS) ===
  [request failed] https://www.nationalpolice.go.ke/allnews?page=0 -> HTTPSConnectionPool(host='www.nationalpolice.go.ke', port=443): Max retries exceeded with url: /allnews?page=0 (Caused by SSLError(SSLCertVerificationError(1, '[SSL: CERTIFICATE_VERIFY_FAILED] certificate verify failed: self-signed certificate in certificate chain (_ssl.c:1010)')))
  -> 0 PSA-style rows collected from National Police Service (NPS)

=== Scraping: National Police Service Commission (NPSC) ===
  page 1: 46 candidate links


National Police Service Commission (NPSC) p1:   0%|          | 0/46 [00:00<?, ?it/s]

  page 2: 46 candidate links


National Police Service Commission (NPSC) p2:   0%|          | 0/46 [00:00<?, ?it/s]

  page 3: 46 candidate links


...

  page 14: 46 candidate links


National Police Service Commission (NPSC) p14:   0%|          | 0/46 [00:00<?, ?it/s]

  page 15: 46 candidate links


National Police Service Commission (NPSC) p15:   0%|          | 0/46 [00:00<?, ?it/s]

  -> 0 PSA-style rows collected from National Police Service Commission (NPSC)

=== Scraping: Kenya Red Cross Society ===
  [request failed] https://redcross.or.ke/news/page/1/ -> 404 Client Error: Not Found for url: https://redcross.or.ke/news/page/1/
  -> 0 PSA-style rows collected from Kenya Red Cross Society

=== Scraping: Ministry of Roads and Transport ===
  [request failed] https://www.transport.go.ke/press-statements/page/1/ -> 404 Client Error: Not Found for url: https://www.transport.go.ke/press-statements/page/1/
  -> 0 PSA-style rows collected from Ministry of Roads and Transport

=== Scraping: Kenya News Agency (KNA) ===
  [request failed] https://www.kenyanews.go.ke/category/security/page/1/ -> HTTPSConnectionPool(host='www.kenyanews.go.ke', port=443): Max retries exceeded with url: /category/security/page/1/ (Caused by SSLError(SSLCertVerificationError(1, '[SSL: CERTIFICATE_VERIFY_FAILED] certificate verify failed: self-signed certificate in certificate chain (_ssl.c:101

## 10. From raw scraped rows to the final dataset schema

Per the project brief the target schema is:
`PSA_ID, Domain, English, Kiswahili, Target Languages (placeholders), Source, Date, Metadata`

Your sample CSV (`PSA_KE_Final.csv`) uses a slightly different, flatter schema:
`PSA_Id, Domain, Class, English, Kiswahili, Ekegusii, Dholuo, Somali`

The cell below reconciles both: it keeps the **CSV's flat per-language columns** (useful for
downstream MT training, matches your sample) but adds the **PDF's `Source`, `Date`, and
`Metadata`** columns, plus a `Sub_Category` column from the Security & Safety taxonomy and a
`Class` column (always `"PSA"` for now, mirroring the sample).

`Kiswahili`, `Ekegusii`, `Dholuo`, and `Somali` are created as **empty placeholders** — per your
instruction, translation (manual or MT-assisted) is a later step, not part of this scraping
notebook.


In [ ]:
# 11. Build the final DataFrame
def build_dataframe(raw_rows, start_id=1):
    df = pd.DataFrame(raw_rows)
    if df.empty:
        print("No rows collected yet -- run the crawl cell above in an internet-connected "
              "environment first.")
        return df

    df["_norm"] = df["English"].str.lower().str.strip()
    df = df.drop_duplicates(subset="_norm").drop(columns="_norm").reset_index(drop=True)

    df.insert(0, "PSA_ID", range(start_id, start_id + len(df)))
    df.insert(1, "Domain", DOMAIN)
    df.insert(2, "Class", "PSA")

    for lang_col in ["Kiswahili", "Ekegusii", "Dholuo", "Somali"]:
        df[lang_col] = ""  # placeholder for manual/MT translation step

    df["Metadata"] = df.apply(
        lambda r: json.dumps({"source_url": r["URL"], "collected_at": datetime.utcnow().isoformat()}),
        axis=1,
    )

    final_cols = [
        "PSA_ID", "Domain", "Class", "Sub_Category",
        "English", "Kiswahili", "Ekegusii", "Dholuo", "Somali",
        "Source", "Date", "Metadata",
    ]
    return df[final_cols]

df_security_safety = build_dataframe(raw_rows)
# df_security_safety.head(10)


No rows collected yet -- run the crawl cell above in an internet-connected environment first.


In [ ]:
# 12. Quick EDA / sanity checks (per Week 1 deliverable: dataset summary stats + samples)
def summarize(df):
    print(f"Total rows: {len(df)}")
    print(f"\nRows per Source:\n{df['Source'].value_counts()}")
    print(f"\nRows per Sub_Category:\n{df['Sub_Category'].value_counts()}")
    print(f"\nDate range: {df['Date'].min()} to {df['Date'].max()}")
    print(f"\nAvg. English sentence length (chars): {df['English'].str.len().mean():.1f}")
    print("\nSample rows:")
    display(df.sample(min(5, len(df)))[["PSA_ID", "Sub_Category", "English", "Source", "Date"]])

# summarize(df_security_safety)


In [ ]:
# 13. Save to CSV (columns aligned so this can later be concatenated with the other
# domain notebooks -- Health, Agriculture, Education, Governance -- into one master dataset)
OUTPUT_PATH = "psa_security_safety_scraped.csv"

df_security_safety.to_csv(OUTPUT_PATH, index=False)
print(f"Saved {len(df_security_safety)} rows to {OUTPUT_PATH}")

Saved 0 rows to psa_security_safety_scraped.csv


## 14. Known limitations & challenges (for your Week 1 report)

- **2,000-row target vs. real availability**: These 10 official sources will very likely **not**
  organically contain 2,000 *distinct, genuinely PSA-shaped* Security & Safety sentences —
  government/NGO news sites publish on the order of dozens to low hundreds of relevant posts
  per year. Realistic ways to close the gap once you've run this and seen the real count:
  - Increase `MAX_PAGES_PER_SOURCE` to pull deeper archives / older years.
  - Add more sources (county government sites, NDMA, Kenya Forest Service fire alerts,
    Communications Authority consumer alerts, county disaster units, more NGOs — WHO Kenya,
    UNICEF Kenya press pages, Star/Nation/Standard "security" tag archives).
  - Use the Wayback Machine (`web.archive.org/web/*/ntsa.go.ke/news/*`) to pull historical
    snapshots of pages that have since changed.
  - Relax `looks_like_psa()` slightly, or have your native-speaker manual-curation pass (Week 1
    checklist item) contribute the remainder — this was always meant to be a **hybrid**
    (scraping + manual) pipeline per the brief, so scraping alone falling short of 2,000 is
    expected, not a failure.
- **Selector fragility**: gov/NGO sites change their HTML without notice and several are on
  shared CMSs (WordPress, Drupal) with inconsistent markup between pages. Always run
  `inspect_source()` on a source before trusting it, and expect to hand-tune 1-2 selectors per
  source.
- **robots.txt / rate limits**: the pipeline checks `robots.txt` before every request and waits
  1.5–3.5s between requests. If a source disallows scraping entirely, it will be skipped and
  logged — note that in your report as a source you had to drop or supplement manually.
- **English-only filter**: `is_english()` drops non-English candidate sentences (Kiswahili
  headlines etc.) since this notebook only populates the `English` column. If a source is
  Kiswahili-first, its Kiswahili content is currently discarded — worth revisiting once you
  reach the translation step, since usable Kiswahili source text could seed that column instead
  of starting it from scratch.
